In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
euro_govs = pd.read_csv('gov_fut.csv')

In [3]:
euro_govs_isin = tuple(euro_govs['isin'].unique())

In [ ]:
# sample: Sep 2024 Bund future (FGBL) on one trading day
sample_isin = 'DE000F0FSQN3'
sample_date = '2024-07-15'

In [ ]:
# Eurex CCP LEIs
eurex_lei = ('529900LN3S50JPU47S06', '529900VIIHQECIS4ET47', '529900UT4DG0LG5R9O07')

In [ ]:
# step 2: sector, with Eurex overridden to CCP
query = f"""
WITH base AS (
    SELECT
        e.tec_ruti,
        e.reference_period,
        e.leg1_direction,
        e.leg1_reporting_cpty_id,
        e.leg1_other_cpty_id,
        CASE WHEN e.leg1_reporting_cpty_id IN {eurex_lei} THEN 'CCP' ELSE s_rep.sector END AS rep_sector,
        CASE WHEN e.leg1_other_cpty_id     IN {eurex_lei} THEN 'CCP' ELSE s_oth.sector END AS oth_sector
    FROM crp_emir_refit_ecb.emir_refit_ecb_trade_states_deduplicated e
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_rep
        ON e.leg1_reporting_cpty_id = s_rep.lei
    LEFT JOIN xlab_ecb_prj_sftds_cb_common.pth_sector s_oth
        ON e.leg1_other_cpty_id = s_oth.lei
    WHERE e.reference_period = '{sample_date}'
        AND e.leg1_product_isin = '{sample_isin}'
        AND e.leg1_direction IN ('BYER', 'SLLR')
)
SELECT
    tec_ruti,
    reference_period,
    CASE WHEN leg1_direction = 'BYER' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS buyer_id,
    CASE WHEN leg1_direction = 'BYER' THEN rep_sector             ELSE oth_sector             END AS buyer_sector,
    CASE WHEN leg1_direction = 'SLLR' THEN leg1_reporting_cpty_id ELSE leg1_other_cpty_id END AS seller_id,
    CASE WHEN leg1_direction = 'SLLR' THEN rep_sector             ELSE oth_sector             END AS seller_sector
FROM base
"""

df = pd.read_sql_query(query, cnxn)
print(df.shape)
print(pd.crosstab(df['buyer_sector'], df['seller_sector']))
df.head()